In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob, os, sparse, sys, warnings, yaml, vcf, pickle, shutil, subprocess
import scipy.optimize

os.chdir("../")
who_variants = pd.read_csv("./data_processing/data_utils/WHO_catalog_V2.csv", header=[2]).reset_index(drop=True)

coll_2014 = pd.read_csv("./data_processing/data_utils/coll2014_SNP_scheme.tsv", sep="\t")
freschi_2020 = pd.read_csv("./data_processing/data_utils/freschi2020_SNP_scheme.tsv", sep="\t")
lineages_matrix = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/lineage_matrix_Coll2014.csv", index_col=[0])

sys.path.append("utils")
from data_utils import *
from analysis_utils import *
from inSilicoMut_utils import *
import seaborn as sns
import scipy.stats as st
plt.rcParams['figure.dpi'] = 150
import statsmodels.stats.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.linear_model import Ridge, RidgeCV, LinearRegression

from Bio import SeqIO
warnings.filterwarnings("ignore")

data_dir = "/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs"
results_dir = "/n/data1/hms/dbmi/farhat/Sanjana/CNN_results"

#cc_df = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/criticalConcentrations_updated.csv")
cc_df = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/MIC/critical_concentrations_all.csv")

drug_abbr_dict = {"Delamanid": "DLM",
                  "Bedaquiline": "BDQ",
                  "Clofazimine": "CFZ",
                  "Ethionamide": "ETO",
                  "Linezolid": "LZD",
                  "Moxifloxacin": "MXF",
                  "Capreomycin": "CAP",
                  "Amikacin": "AMK",
                  "Pretomanid": "PMD",
                  "Pyrazinamide": "PZA",
                  "Kanamycin": "KAN",
                  "Levofloxacin": "LFX",
                  "Streptomycin": "STM",
                  "Ethambutol": "EMB",
                  "Isoniazid": "INH",
                  "Rifampicin": "RIF"
                 }

abbr_drug_dict = {value: key for key, value in drug_abbr_dict.items()}

model_loci = pd.read_csv("./data_processing/data_utils/drug_loci.csv")
model_loci[['Start', 'End']] = model_loci[['Start', 'End']].astype(int)

cryptic_wgs_metadata = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/cryptic_WGS_metadata.csv")
df_cryptic = pd.read_csv("MIC_data/CRyPTIC_reuse_table_20231208.csv")
df_geno = pd.read_csv("./data_processing/samples_pass_geno_QC.csv")

cc_dict = {'RIF': 0.5, 'INH': 0.2, 'EMB': 5, 'PZA': 100, 'ETO': 5, 'MXF': 0.5, 'LFX': 1, 'BDQ': 0.25, 'LZD': 1, 'CFZ': 1, 'DLM': 0.016}
drugs_lst = ['BDQ', 'EMB', 'ETO', 'INH', 'LFX', 'MXF', 'PZA', 'RIF']

isolate_variants = pd.read_csv(os.path.join(os.path.dirname(data_dir), "isolate_WHO_catalog_variants.csv")).dropna(subset='variant').reset_index(drop=True)

/tmp/ipykernel_20612/3974794963.py:8: DtypeWarning: Columns (36,37,99,100,102,103,106,108,112) have mixed types. Specify dtype option on import or set low_memory=False.
  who_variants = pd.read_csv("./data_processing/data_utils/WHO_catalog_V2.csv", header=[2]).reset_index(drop=True)


# 1. Get Binary Bedaquiline data from the WHO Catalog

In [ ]:
phenos_binary

In [67]:
sample_ids_mapping = pd.read_csv("~/who-analysis/lineages/combined_lineages_samples.csv")
phenos_binary = pd.read_csv("./analysis/BDQ_WHO_catalog_phenos_binary.csv")

In [93]:
df_BDQ_before_QC = pd.read_csv(f"{data_dir}/BDQ/combined_MIC.csv")

BDQ_R_binary = sample_ids_mapping.merge(phenos_binary.query("phenotype==1"), left_on='Sample_ID', right_on='sample_id')[['BioSample', 'Sample_Name', 'Lineage', 'phenotypic_category']]
print(f"{len(BDQ_R_binary.dropna(subset='BioSample'))}/{len(BDQ_R_binary)} resistant samples are public")
BDQ_R_binary = BDQ_R_binary.dropna(subset='BioSample')

BDQ_R_binary['MIC'] = BDQ_R_binary['BioSample'].isin(df_BDQ_before_QC.ROLLINGDB_ID).astype(int)
print(f"{BDQ_R_binary.MIC.sum()} samples are already in the MIC dataset")
num_WHO_datapoints = len(BDQ_R_binary.query("phenotypic_category=='WHO'"))
print(f"{num_WHO_datapoints} samples are in the WHO dataset")

# there's one BioSample: SAMN11603930 with many different WGS runs and 1 different lineage case
BDQ_R_binary = BDQ_R_binary.query("MIC==0 & phenotypic_category=='WHO' & ~BioSample.str.contains('_')").reset_index(drop=True)

987/1037 resistant samples are public
95 samples are already in the MIC dataset
880 samples are in the WHO dataset


In [156]:
BDQ_R_binary.Lineage.value_counts()

Lineage
2      379
4      258
3      158
1       70
1,2      1
2,3      1
Name: count, dtype: int64

In [157]:
BDQ_R_binary.query("Lineage.str.contains(',')")

,BioSample,Sample_Name,Lineage,phenotypic_category,MIC
497,SAMEA110420255,SAMEA110420255,"1,2",WHO,0
609,SAMEA110420142,SAMEA110420142,"2,3",WHO,0


In [131]:
BDQ_binary_samples_metadata = pd.read_csv("./analysis/BDQ_binary_samples_metadata.csv").query("Platform=='ILLUMINA'").reset_index(drop=True)
BDQ_binary_samples_metadata.Query.nunique(), BDQ_binary_samples_metadata.BioSample.nunique(), BDQ_binary_samples_metadata.Run.nunique()

(867, 867, 901)

In [132]:
# 34 samples had GenoScreen DeepLex and WGS done, and they are all named <sample_name>-GS or <sample_name>-WGS, so sort and drop
BDQ_binary_samples_metadata = BDQ_binary_samples_metadata.sort_values(["BioSample", "LibraryName"]).drop_duplicates('BioSample', keep='last')
BDQ_binary_samples_metadata.Query.nunique(), BDQ_binary_samples_metadata.BioSample.nunique(), BDQ_binary_samples_metadata.Run.nunique()

(867, 867, 867)

In [137]:
len(BDQ_binary_samples_metadata.query("LibraryName.str.endswith('-GS')")), len(BDQ_binary_samples_metadata.query("LibraryName.str.endswith('-WGS')"))

(0, 34)

In [141]:
df_megapipe = BDQ_binary_samples_metadata[['BioSample', 'Run']].drop_duplicates(subset='BioSample')
df_megapipe['FQ'] = 1

In [143]:
df_megapipe.to_csv("~/Mtb_Megapipe/BDQ_binary_samples.tsv", sep='\t', header=None, index=False)

# 1. Make duplicates of isolates with Group 1-2 variants that the model missed

Basically all missense variants

In [2]:
all_drugs_insilico_pred = pd.read_csv("supplement/insilico_pred_all.csv")

In [3]:
df_BDQ = pd.read_csv(os.path.join(data_dir, "BDQ", "data_for_model.csv"))

variants_in_training_isolates = isolate_variants.merge(df_BDQ.query("Span_CC==0")).query("drug=='Bedaquiline' & confidence in ['1) Assoc w R', '2) Assoc w R - Interim'] & AF > 0.75").variant.unique()

In [4]:
missed_Group12_variants = all_drugs_insilico_pred.query("drug=='Bedaquiline' & variant in @variants_in_training_isolates & pred_MIC < 0.25").variant.values
len(missed_Group12_variants)

10

In [5]:
# get the isolates that have these variants
missed_Group12_variants_isolate_MICs = isolate_variants.merge(df_BDQ.query("Span_CC==0"))[['ROLLINGDB_ID', 'variant', 'BDQ_lower_bound', 'BDQ_midpoint', 'BDQ_upper_bound', 'Lineage', 'DB_OF_ORIGIN']].query("variant in @missed_Group12_variants").drop_duplicates()

In [62]:
# # take isolate with the maximum MIC for each variant and make 10 copies of it
# # missed_Group12_variants_isolate_MICs.groupby("variant")[['BDQ_lower_bound', 'BDQ_midpoint', 'BDQ_upper_bound']].max().sort_values("BDQ_lower_bound")
# samples_to_multiply = missed_Group12_variants_isolate_MICs.sort_values(["variant", 'BDQ_lower_bound']).drop_duplicates("variant", keep='last').query("BDQ_lower_bound >= 0.12").sort_values("BDQ_lower_bound").ROLLINGDB_ID.values

# print(len(samples_to_multiply))

# missed_Group12_variants_isolate_MICs.sort_values(["variant", 'BDQ_lower_bound']).drop_duplicates("variant", keep='last').query("BDQ_lower_bound >= 0.12").sort_values("BDQ_lower_bound")

9


,ROLLINGDB_ID,variant,BDQ_lower_bound,BDQ_midpoint,BDQ_upper_bound,Lineage,DB_OF_ORIGIN
410298,SAMEA8718648,Rv0678_p.Asn70Asp,0.12,0.185,0.25,2,CRyPTIC
161702,SAMEA7541691,Rv0678_p.Cys46Arg,0.12,0.185,0.25,4,CRyPTIC
321351,SAMEA7562380,Rv0678_p.Leu117Arg,0.12,0.185,0.25,3,CRyPTIC
363003,SAMEA7563307,Rv0678_p.Met146Thr,0.12,0.185,0.25,4,CRyPTIC
415809,SAMEA8718804,Rv0678_p.Ala36Val,0.25,0.375,0.50,2,CRyPTIC
42510,SAMEA1102223,Rv0678_p.Gly121Arg,0.25,0.375,0.50,3,CRyPTIC
414638,SAMEA8718769,Rv0678_p.Ile67Ser,0.25,0.375,0.50,2,CRyPTIC
377675,SAMEA7563608,atpE_p.Ala63Pro,1.00,1.000,inf,2,CRyPTIC
76144,SAMEA7524903,atpE_p.Glu61Asp,1.00,1.500,2.00,2,CRyPTIC


In [40]:
all_drugs_insilico_pred.query("drug=='Bedaquiline' & variant in @variants_in_training_isolates").sort_values("pred_MIC")

,variant,log2_pred_MIC,pred_MIC,effect,CC,Locus,confidence,ref_pred,fold_change,log2_fold_change,gene,drug
63,Rv0678_p.Asn70Asp,-5.052226,0.030139,missense_variant,0.25,Rv0678,2) Assoc w R - Interim,0.029826,1.010489,0.015054,Rv0678,Bedaquiline
130,Rv0678_p.Leu32Ser,-5.039315,0.030410,missense_variant,0.25,Rv0678,2) Assoc w R - Interim,0.029826,1.019573,0.027966,Rv0678,Bedaquiline
206,atpE_p.Ala63Pro,-5.038499,0.030427,missense_variant,0.25,atpE,2) Assoc w R - Interim,0.029826,1.020150,0.028781,atpE,Bedaquiline
74,Rv0678_p.Cys46Arg,-5.011257,0.031007,missense_variant,0.25,Rv0678,2) Assoc w R - Interim,0.029826,1.039596,0.056024,Rv0678,Bedaquiline
212,atpE_p.Glu61Asp,-5.003971,0.031164,missense_variant,0.25,atpE,2) Assoc w R - Interim,0.029826,1.044860,0.063310,atpE,Bedaquiline
26,Rv0678_p.Ala36Val,-4.915077,0.033145,missense_variant,0.25,Rv0678,2) Assoc w R - Interim,0.029826,1.111265,0.152203,Rv0678,Bedaquiline
121,Rv0678_p.Ile67Ser,-4.853150,0.034598,missense_variant,0.25,Rv0678,2) Assoc w R - Interim,0.029826,1.160005,0.214130,Rv0678,Bedaquiline
146,Rv0678_p.Met146Thr,-4.036098,0.060956,missense_variant,0.25,Rv0678,2) Assoc w R - Interim,0.029826,2.043698,1.031182,Rv0678,Bedaquiline
103,Rv0678_p.Gly121Arg,-3.572095,0.084080,missense_variant,0.25,Rv0678,1) Assoc w R,0.029826,2.819003,1.495185,Rv0678,Bedaquiline
124,Rv0678_p.Leu117Arg,-3.444499,0.091855,missense_variant,0.25,Rv0678,1) Assoc w R,0.029826,3.079681,1.622781,Rv0678,Bedaquiline


In [44]:
# df_BDQ_with_augmentation = df_BDQ.copy()

# num_multiples = 10

# for sample in samples_to_multiply:

#     df_new = pd.concat([df_BDQ_with_augmentation.query("ROLLINGDB_ID==@sample")]*10,ignore_index=True)
#     df_new['ROLLINGDB_ID'] = df_new['ROLLINGDB_ID'] + '_' + df_new.index.astype(str)
#     df_new['category'] = 'train_set'
#     df_new['Span_CC'] = 0
#     df_BDQ_with_augmentation = pd.concat([df_BDQ_with_augmentation, df_new])

#     for new_name in df_new.ROLLINGDB_ID.values:
#         shutil.copy(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{sample}.vcf", f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{new_name}.vcf")
        
# df_BDQ_with_augmentation = df_BDQ_with_augmentation.reset_index(drop=True)

In [64]:
len(df_BDQ_with_augmentation), len(df_BDQ)

(9228, 9138)

In [65]:
df_BDQ_with_augmentation.to_csv(f"{data_dir}/BDQ/data_for_model_augmented.csv", index=False)

In [65]:
all_drugs_insilico_pred.query("drug=='Bedaquiline' & confidence=='1) Assoc w R'")

,variant,log2_pred_MIC,pred_MIC,effect,CC,Locus,confidence,ref_pred,fold_change,log2_fold_change,gene,drug
103,Rv0678_p.Gly121Arg,-3.572095,0.084080,missense_variant,0.25,Rv0678,1) Assoc w R,0.029826,2.819003,1.495185,Rv0678,Bedaquiline
124,Rv0678_p.Leu117Arg,-3.444499,0.091855,missense_variant,0.25,Rv0678,1) Assoc w R,0.029826,3.079681,1.622781,Rv0678,Bedaquiline


In [70]:
df_BDQ_with_augmentation

,ROLLINGDB_ID,BDQ_MEDIA,BDQ_lower_bound,BDQ_midpoint,BDQ_upper_bound,MEDIA,DB_OF_ORIGIN,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,F2,Lineage,log2_F2,Span_CC,Binary,Stratify,category
0,SAMEA104362049,7H11,0.015,0.0225,0.03,UKMYC,CRyPTIC,4.1.1,4.1.i1.2.1,"xtype,canetti",NaN,4,0.018976,4,-5.719682,0,0.0,4-0,test_set
1,SAMEA104362050,7H11,0.030,0.0450,0.06,UKMYC,CRyPTIC,4.1.2.1,4.1.i1.1.1.1,haarlem,NaN,4.1.2/Haarlem,0.011908,4,-6.391939,0,0.0,4-0,test_set
2,SAMEA104362060,7H11,0.015,0.0225,0.03,UKMYC,CRyPTIC,4.3.3,4.2.1.2.1.1.i4.1,lam,NaN,4.3/LAM,0.007257,4,-7.106395,0,0.0,4-0,test_set
3,SAMEA104362065,7H11,0.015,0.0225,0.03,UKMYC,CRyPTIC,4.1.2.1,4.1.i1.1.1.1,haarlem,NaN,4.1.2/Haarlem,0.008932,4,-6.806869,0,0.0,4-0,test_set
4,SAMEA104362069,7H11,0.060,0.0900,0.12,UKMYC,CRyPTIC,4.1.2.1,4.1.i1.1.1.1,"haarlem,westafrican1",NaN,4.1.2/Haarlem,0.010520,4,-6.570780,0,0.0,4-0,test_set
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9223,SAMEA7524903_5,7H11,1.000,1.5000,2.00,UKMYC,CRyPTIC,2.2.1,2.2.1.1.1,"beijing,mungi,canetti","lin2.2.1,asian_african_2",NaN,0.010499,2,-6.573559,0,1.0,2-1,train_set
9224,SAMEA7524903_6,7H11,1.000,1.5000,2.00,UKMYC,CRyPTIC,2.2.1,2.2.1.1.1,"beijing,mungi,canetti","lin2.2.1,asian_african_2",NaN,0.010499,2,-6.573559,0,1.0,2-1,train_set
9225,SAMEA7524903_7,7H11,1.000,1.5000,2.00,UKMYC,CRyPTIC,2.2.1,2.2.1.1.1,"beijing,mungi,canetti","lin2.2.1,asian_african_2",NaN,0.010499,2,-6.573559,0,1.0,2-1,train_set
9226,SAMEA7524903_8,7H11,1.000,1.5000,2.00,UKMYC,CRyPTIC,2.2.1,2.2.1.1.1,"beijing,mungi,canetti","lin2.2.1,asian_african_2",NaN,0.010499,2,-6.573559,0,1.0,2-1,train_set


# Additional Missense Variants to Add

Restrict to isolates with a 7H11 reported MIC (not just fold change):

Rv0678_p.Trp42Arg: 0.0625-0.12 µg/mL
Rv0678_p.Trp42Arg: 0.0625-0.12 µg/mL
Rv0678_c.139_140insG: 0.12-0.25 µg/mL
Rv0678_c.140_141insG: 0.12-0.25 µg/mL
Rv0678_p.Ser53Pro: 0.25-0.5 µg/mL
Rv0678_p.Ser53Pro: 0.25-0.5 µg/mL
Rv0678_p.Ser53Pro: 0.25-0.5 µg/mL
Rv0678_p.Ser53Leu: 0.25-0.5 µg/mL
Rv0678_p.Ser53Leu: 0.25-0.5 µg/mL
Rv0678_c.274_275insA: 0.5–1 µg/mL
Rv0678_c.274_275insA: 0.5–1 µg/mL
Rv0678_p.Ser63Arg + Rv0678_p.Arg50Trp: 0.24-0.48 µg/mL
Rv0678_p.Ser63Gly + Rv0678_p.Arg50Trp: 0.24-0.48 µg/mL
Rv0678_p.Ser63Arg + Rv0678_p.Arg50Trp: 0.24-0.48 µg/mL
Rv0678_p.Ser63Gly + Rv0678_p.Arg50Trp: 0.24-0.48 µg/mL

Additional variants with only fold changes reported:

atpE_p.Asp28Val: 0.25-0.5 µg/mL
atpE_p.Asp28Pro: 
atpE_p.Asp28Asn: 0.06-0.12 µg/mL
atpE_p.Glu61Asp: 
atpE_p.Ile66Met: 0.06-0.125 µg/mL

Rv0678_p.Leu142Arg: 0.12-0.25 µg/mL
Rv0678_p.Leu142Arg + atpE_p.Ala63Val: 0.5-1 µg/mL
Rv0678_p.Gly121Glu: 0.12-0.25 µg/mL

In [9]:
variants = ['Rv0678_p.Trp42Arg', 
            'Rv0678_p.Trp42Arg',
            'Rv0678_c.139insG',
            'Rv0678_c.140insG',
            'Rv0678_p.Ser53Pro',
            'Rv0678_p.Ser53Pro',
            'Rv0678_p.Ser53Pro',
            'Rv0678_p.Ser53Leu',
            'Rv0678_p.Ser53Leu',
            'Rv0678_c.274insA',
            'Rv0678_c.274insA',
            'Rv0678_p.Ser63Arg',
            'Rv0678_p.Ser63Gly',
            'Rv0678_p.Arg50Trp',
            'Rv0678_p.Arg50Trp'
           ]

# MIC_lb = []
# MIC_ub = []

df_variants = pd.DataFrame()

df_variants['variant'] = variants
df_variants['gene'] = df_variants['variant'].str.split('_').str[0]
df_variants['mutation'] = df_variants['variant'].str.replace('Rv0678_', '')

df_variants.loc[df_variants["variant"].str.contains('ins'), 'effect'] = 'insertion'
df_variants['effect'] = df_variants['effect'].replace('nan', 'missense_variant')
df_variants['VCF_file'] = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 12, 13]

In [12]:
df_variants

,variant,gene,mutation,effect,VCF_file
0,Rv0678_p.Trp42Arg,Rv0678,p.Trp42Arg,missense_variant,1
1,Rv0678_p.Trp42Arg,Rv0678,p.Trp42Arg,missense_variant,2
2,Rv0678_c.139insG,Rv0678,c.139insG,insertion,3
3,Rv0678_c.140insG,Rv0678,c.140insG,insertion,4
4,Rv0678_p.Ser53Pro,Rv0678,p.Ser53Pro,missense_variant,5
5,Rv0678_p.Ser53Pro,Rv0678,p.Ser53Pro,missense_variant,6
6,Rv0678_p.Ser53Pro,Rv0678,p.Ser53Pro,missense_variant,7
7,Rv0678_p.Ser53Leu,Rv0678,p.Ser53Leu,missense_variant,8
8,Rv0678_p.Ser53Leu,Rv0678,p.Ser53Leu,missense_variant,9
9,Rv0678_c.274insA,Rv0678,c.274insA,insertion,10


In [18]:
df_for_VCF = get_data_for_synthetic_VCF(df_variants)

In [19]:
def create_synthetic_VCF_files(df, out_fName, vcf_dir):

    # assert len(df) == df.variant.nunique()
    
    # create a header section
    header = '##fileformat=VCFv4.1\n'
    header += "##contig=<ID=NC_000962.3,length=4411532>\n"
    header += '#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tSample\n'

    print(f"Creating synthetic VCF files for {df['mutation'].nunique()} mutations")

    with open(out_fName, 'w+') as out_file:
         
        # ITERATE through each mutation. N_mutations = N_files to be created. 
        # There can be multiple single site variants to make for a given mutation
        for single_VCF in df.VCF_file.unique():

            combined_mutation_str = []

            for _, row in df.query("VCF_file==@single_VCF").drop_duplicates().iterrows():
            
                # need to remove special characters
                mutation_str = row['variant'].replace('.', '_').replace('*', '+')
                combined_mutation_str.append(mutation_str)

            combined_mutation_str = '-'.join(combined_mutation_str)
    
            # absolute VCF file path
            vcf_fName = f"{vcf_dir}/{combined_mutation_str}.vcf"
            print(vcf_fName)
            
            out_file.write(vcf_fName + "\n")
                            
            # create a VCF file for the mutation
            with open(vcf_fName, 'w+') as vcf_file:

                # write VCF file header
                vcf_file.write(header)

                for _, row in df.query("VCF_file==@single_VCF").iterrows():

                    # write the variant that causes the mutation
                    vcf_file.write('\t'.join(['NC_000962.3', str(row["POS"]), '.', row["REF"], row["ALT"], '.', 'PASS', '.', 'GT', '1/1']) + '\n')

In [20]:
create_synthetic_VCF_files(df_for_VCF, "/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/BDQ/augmented_VCF_names.txt", "/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF")

Creating synthetic VCF files for 9 mutations
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Trp42Arg.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Trp42Arg.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_c_139insG.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_c_140insG.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Ser53Pro.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Ser53Pro.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Ser53Pro.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Ser53Leu.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Ser53Leu.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_c_274insA.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_c_274insA.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Ser63Arg-Rv0678_p_Arg50Trp.vcf
/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/Rv0678_p_Ser63Gly-Rv0678_p_Arg50Trp.vcf


In [ ]:
new_paths = [f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/BDQ"]

In [ ]:
df_BDQ_with_augmentation.loc[-1, ['ROLLINGDB_ID', 
                                  'BDQ_MEDIA', 
                                  'BDQ_lower_bound', 
                                  'BDQ_midpoint', 
                                  'BDQ_upper_bound', 
                                  'MEDIA', 
                                  'DB_OF_ORIGIN']] = ['Rv0678_p_Trp42Arg',
                                                      'Rv0678_c_139insG',
                                                      'Rv0678_c_140insG',
                                                      'Rv0678_p_Ser53Pro',
                                                      
                                                     ]